# ECCE 2027 Smoke Test -- EVAL ONLY (Kaggle): Dhaka Fold 0 (E1.0)

Runs in a brand-new container, separate from `train_e1_0.ipynb`, so the GPU is
guaranteed clean -- this sidesteps the CUDA OOM that a combined train+eval
process kept hitting on the T4 x2 tier no matter how much cleanup ran between
the two phases.

**Attach two inputs before running** (Add Input, right sidebar):
1. `badodd-ecce2027-bundle` (same dataset as before -- images + overlay + coco_gt).
2. `train_e1_0` -- **your own `train_e1_0.ipynb` notebook**, added as an input
   (Add Input -> search your notebooks -> select it). Kaggle exposes its
   `/kaggle/working` output under `/kaggle/input/train-e1-0/` (or similar slug)
   automatically -- no need to manually package it as a dataset. It must have a
   completed, saved version first.

Settings: Accelerator = GPU T4 x2, Internet = ON.

In [ ]:
# ==============================================================================
# 1. HARDWARE & ENVIRONMENT VERIFICATION (fresh container -- GPU should start clean)
# ==============================================================================
import os, sys, time, glob, json, shutil, subprocess

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from pathlib import Path
import torch

PINNED_ULTRALYTICS = "8.4.155"

print('Python version:', sys.version)
print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
print('CUDA device count (should be 1 after masking):', torch.cuda.device_count())

subprocess.run("nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free "
               "--format=csv", shell=True)

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name} ({total_vram:.2f} GB VRAM)')
    allocated_at_start = torch.cuda.memory_allocated(0) / 1e9
    print(f'GPU memory allocated at notebook start: {allocated_at_start:.3f} GB (should be ~0)')
else:
    print('WARNING: No GPU detected! Enable GPU accelerator in notebook settings.')

subprocess.run(f"pip install -q ultralytics=={PINNED_ULTRALYTICS} pycocotools", shell=True, check=True)
import ultralytics
assert ultralytics.__version__ == PINNED_ULTRALYTICS, (
    f"Ultralytics version drift: installed {ultralytics.__version__}, expected {PINNED_ULTRALYTICS}"
)
print('Ultralytics version:', ultralytics.__version__)

from ultralytics import YOLO

In [ ]:
# ==============================================================================
# 2. DATASET & OVERLAY DISCOVERY -- FAIL HARD on any missing image
# ==============================================================================
print('=== Scanning /kaggle/input for Dataset & Overlay ===')

badodd_zips = glob.glob('/kaggle/input/**/badodd.zip', recursive=True)
if badodd_zips and not glob.glob('/kaggle/input/**/*.jpg', recursive=True):
    print(f'Found badodd.zip at {badodd_zips[0]}. Extracting to /kaggle/working/badodd_images...')
    os.makedirs('/kaggle/working/badodd_images', exist_ok=True)
    subprocess.run(f"unzip -q {badodd_zips[0]} -d /kaggle/working/badodd_images", shell=True, check=True)
    IMAGE_SEARCH_ROOT = '/kaggle/working/badodd_images'
else:
    IMAGE_SEARCH_ROOT = '/kaggle/input'

all_input_imgs = [p for p in glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.jpg', recursive=True) +
                        glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.png', recursive=True)
                   if not os.path.basename(p).startswith('._')]
img_lookup = {os.path.basename(p): p for p in all_input_imgs}
print(f'Indexed {len(img_lookup)} images from {IMAGE_SEARCH_ROOT}.')
assert len(img_lookup) >= 10000, (
    f"Expected ~10,032 BadODD images, found only {len(img_lookup)} -- "
    f"is the bundle dataset attached?"
)

overlay_roots = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'splits' in dirs and 'labels' in dirs:
        overlay_roots.append(root)

overlay_zips = glob.glob('/kaggle/input/**/badodd_ecce2027_overlay*.zip', recursive=True)
overlay_src = None
if overlay_roots:
    overlay_src = overlay_roots[0]
    print(f'Found overlay directory at: {overlay_src}')
elif overlay_zips:
    print(f'Found overlay zip at: {overlay_zips[0]}. Unzipping to /kaggle/working/overlay...')
    subprocess.run(f"unzip -q {overlay_zips[0]} -d /kaggle/working/overlay", shell=True, check=True)
    overlay_src = '/kaggle/working/overlay'
else:
    raise FileNotFoundError(
        'Could not locate the ECCE 2027 overlay package in /kaggle/input! '
        'Attach the bundle dataset.'
    )

assert os.path.isdir(os.path.join(overlay_src, 'coco_gt'))
assert os.path.isdir(os.path.join(overlay_src, 'splits'))

WORK_DATA = '/kaggle/working/data'
WORK_IMAGES = os.path.join(WORK_DATA, 'images')
WORK_SPLITS = os.path.join(WORK_DATA, 'splits')
os.makedirs(WORK_IMAGES, exist_ok=True)
os.makedirs(WORK_SPLITS, exist_ok=True)

for bname, src_p in img_lookup.items():
    dest = os.path.join(WORK_IMAGES, bname)
    if not os.path.exists(dest):
        try:
            os.symlink(src_p, dest)
        except OSError:
            shutil.copy(src_p, dest)
print(f'Linked {len(img_lookup)} image files.')

# Only need the Chattogram pooled eval manifest resolved, but do it the same
# fail-hard way as the training notebook.
eval_manifest_src = os.path.join(overlay_src, 'splits', 'ctg_pooled_eval.txt')
with open(eval_manifest_src) as f:
    bases = [os.path.basename(l.strip()) for l in f if l.strip()]
resolved = []
missing_here = []
for b in bases:
    p = os.path.join(WORK_IMAGES, b)
    if os.path.exists(p):
        resolved.append(p)
    else:
        missing_here.append(b)
if missing_here:
    raise FileNotFoundError(
        f"ctg_pooled_eval.txt: {len(missing_here)} images missing, e.g. {missing_here[:5]}. "
        f"Do not proceed -- check the attached dataset."
    )
EVAL_MANIFEST = os.path.join(WORK_SPLITS, 'ctg_pooled_eval.txt')
with open(EVAL_MANIFEST, 'w') as f:
    f.writelines(p + '\n' for p in resolved)
print(f'Resolved ctg_pooled_eval.txt: {len(resolved)} / {len(bases)} images present.')

CLASS_NAMES = {
    0: 'auto_rickshaw', 1: 'bicycle', 2: 'bus', 3: 'car', 4: 'cart_vehicle',
    5: 'construction_vehicle', 6: 'motorbike', 7: 'person', 8: 'priority_vehicle',
    9: 'three_wheeler', 10: 'truck',
}

In [ ]:
# ==============================================================================
# 3. LOCATE THE TRAINED CHECKPOINT FROM train_e1_0.ipynb's OUTPUT
# ==============================================================================
ckpt_candidates = glob.glob('/kaggle/input/**/e1_0_smoke_last.pt', recursive=True)
if not ckpt_candidates:
    raise FileNotFoundError(
        "Could not find e1_0_smoke_last.pt under /kaggle/input. Attach train_e1_0.ipynb "
        "as an input (Add Input -> your notebooks -> train_e1_0), and make sure it has "
        "a completed, saved version."
    )
print(f'Checkpoint candidates found ({len(ckpt_candidates)}):')
for c in ckpt_candidates:
    print(f'  {c}  ({os.path.getsize(c)/1e6:.2f} MB)')
if len(ckpt_candidates) > 1:
    print('WARNING: multiple matches -- using the first one. If train_e1_0 was re-run '
          'multiple times, stale copies from earlier versions may be attached too.')
LAST_CKPT = ckpt_candidates[0]
print(f'\nUsing checkpoint: {LAST_CKPT} ({os.path.getsize(LAST_CKPT)/1e6:.2f} MB)')
assert os.path.getsize(LAST_CKPT) < 200e6, (
    f'Checkpoint is {os.path.getsize(LAST_CKPT)/1e6:.1f} MB -- a YOLOv8s last.pt should be '
    f'~45-50 MB. Something is wrong with this file (wrong file matched, or it is a bundled '
    f'multi-model checkpoint) -- do not proceed.'
)

run_info_candidates = glob.glob('/kaggle/input/**/e1_0_smoke_run_info.json', recursive=True)
if run_info_candidates:
    with open(run_info_candidates[0]) as f:
        train_run_info = json.load(f)
    print('\nTraining run info (from train_e1_0.ipynb):')
    print(json.dumps(train_run_info, indent=2))
else:
    train_run_info = None
    print('WARNING: run_info.json not found alongside the checkpoint -- proceeding without it.')

In [ ]:
# ==============================================================================
# 4. EXPORT COCO-JSON PREDICTIONS AT conf=0.001 (CRITICAL)
# ==============================================================================
# Diagnosis from prior runs: the pre-flight single-image test always passes,
# but the full stream_inference loop leaks ~900 MB of ACTIVE (not cached)
# GPU memory per batch of 16 and never releases it -- torch.cuda.empty_cache()
# cannot fix this because it only returns reserved-but-unused memory, and the
# leaked memory here is genuinely still "allocated". Rather than depend on
# pinning down which internal ultralytics/predictor state is retaining it, we
# sidestep it entirely: process images in small chunks and fully destroy +
# reload the model between chunks, so nothing accumulates past a safe ceiling.
import gc

CHUNK_SIZE = 128  # ~8 batches of 16 per chunk -- keeps growth well under ~11 GB headroom

if torch.cuda.is_available():
    print('Sanity check just before loading model:')
    print(f'  torch.cuda.current_device() = {torch.cuda.current_device()}')
    print(f'  torch.cuda.device_count()   = {torch.cuda.device_count()}')
    print(f'  allocated = {torch.cuda.memory_allocated(0)/1e9:.3f} GB')
    print(f'  reserved  = {torch.cuda.memory_reserved(0)/1e9:.3f} GB')
    subprocess.run("nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free "
                   "--format=csv", shell=True)

RESULTS_DIR = '/kaggle/working/results/predictions'
os.makedirs(RESULTS_DIR, exist_ok=True)
OUTPUT_JSON = os.path.join(RESULTS_DIR, 'e1_0_smoke_ctg_pooled_eval_preds_conf0001.json')

with open(EVAL_MANIFEST, 'r') as f:
    eval_images = [l.strip() for l in f if l.strip()]

print(f'\nEvaluating last.pt model on {len(eval_images)} Chattogram evaluation images...')

# --- Pre-flight bisection test: single image, fresh model. Confirmed passing
# in prior runs -- kept as a cheap early-exit sanity check.
_preflight_model = YOLO(LAST_CKPT)
print('\nPre-flight test: running inference on 1 image to isolate the failure mode...')
try:
    _preflight = list(_preflight_model.predict(
        source=[eval_images[0]], conf=0.001, iou=0.7, imgsz=640,
        device=0 if torch.cuda.is_available() else 'cpu',
        batch=1, stream=True, verbose=False, workers=0,
    ))
    if torch.cuda.is_available():
        print(f'Pre-flight OK. allocated={torch.cuda.memory_allocated(0)/1e9:.3f} GB, '
              f'reserved={torch.cuda.memory_reserved(0)/1e9:.3f} GB')
except torch.cuda.OutOfMemoryError:
    print('\n*** OOM ON THE VERY FIRST SINGLE-IMAGE PREDICTION ***')
    print(torch.cuda.memory_summary(device=0, abbreviated=True))
    subprocess.run("nvidia-smi", shell=True)
    raise
del _preflight_model, _preflight
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

coco_predictions = []
class_counts = {cid: 0 for cid in CLASS_NAMES}
total_boxes = 0
t0_infer = time.time()

chunks = [eval_images[i:i + CHUNK_SIZE] for i in range(0, len(eval_images), CHUNK_SIZE)]
print(f'\nProcessing {len(eval_images)} images in {len(chunks)} chunks of up to {CHUNK_SIZE} '
      f'(model reloaded fresh each chunk to prevent cross-chunk memory accumulation)...')

images_done = 0
for chunk_idx, chunk in enumerate(chunks):
    try:
        chunk_model = YOLO(LAST_CKPT)
        results_gen = chunk_model.predict(
            source=chunk,
            conf=0.001,
            iou=0.7,
            imgsz=640,
            device=0 if torch.cuda.is_available() else 'cpu',
            batch=16,
            stream=True,
            verbose=False,
            workers=0,
        )

        for r in results_gen:
            img_stem = Path(r.path).stem
            boxes = r.boxes
            if boxes is not None and len(boxes) > 0:
                xyxy = boxes.xyxy.cpu().numpy()
                confs = boxes.conf.cpu().numpy()
                clss = boxes.cls.cpu().numpy().astype(int)
                for i in range(len(boxes)):
                    x1, y1, x2, y2 = xyxy[i]
                    w = x2 - x1
                    h = y2 - y1
                    cid = int(clss[i])
                    coco_predictions.append({
                        'image_id': img_stem,
                        'category_id': cid,
                        'bbox': [round(float(x1), 2), round(float(y1), 2), round(float(w), 2), round(float(h), 2)],
                        'score': round(float(confs[i]), 5),
                    })
                    total_boxes += 1
                    if cid in class_counts:
                        class_counts[cid] += 1
            del r
        images_done += len(chunk)

        del chunk_model, results_gen
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if torch.cuda.is_available():
            alloc_now = torch.cuda.memory_allocated(0) / 1e9
        else:
            alloc_now = 0.0
        print(f'  chunk {chunk_idx + 1}/{len(chunks)}: {images_done}/{len(eval_images)} images done, '
              f'boxes so far={total_boxes}, allocated after chunk={alloc_now:.3f} GB')

    except torch.cuda.OutOfMemoryError:
        print(f'\n*** OOM DURING CHUNK {chunk_idx + 1}/{len(chunks)} '
              f'(images {images_done}-{images_done + len(chunk)}) ***')
        print(torch.cuda.memory_summary(device=0, abbreviated=True))
        subprocess.run("nvidia-smi", shell=True)
        raise

total_infer_time = time.time() - t0_infer
fps = len(eval_images) / total_infer_time
print(f'\nInference finished in {total_infer_time:.2f} s ({fps:.1f} FPS)')
print(f'Total exported detections at conf>=0.001: {total_boxes}')
print('\nDetections by Class:')
for cid, cname in CLASS_NAMES.items():
    print(f'  [{cid:2d}] {cname:22s}: {class_counts[cid]}')

with open(OUTPUT_JSON, 'w') as f:
    json.dump(coco_predictions, f)
json_size_mb = os.path.getsize(OUTPUT_JSON) / (1024*1024)
print(f'\nExported JSON: {OUTPUT_JSON} ({json_size_mb:.2f} MB)')
assert total_boxes > 0, 'FAIL: zero predictions exported.'

In [ ]:
# ==============================================================================
# 5. PYCOCOTOOLS ALIGNMENT CHECK -- proves category ids / bbox coords / image ids match GT
# ==============================================================================
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

GT_JSON = os.path.join(overlay_src, 'coco_gt', 'ctg_pooled_eval_coco_gt.json')
coco_gt = COCO(GT_JSON)
coco_dt = coco_gt.loadRes(OUTPUT_JSON)

ev = COCOeval(coco_gt, coco_dt, iouType='bbox')
ev.evaluate(); ev.accumulate(); ev.summarize()
mAP50_95 = ev.stats[0]
mAP50 = ev.stats[1]

print(f'\nOverall mAP50: {mAP50:.4f} | mAP50-95: {mAP50_95:.4f}')

print('\nPer-class AP50:')
cat_ids = ev.params.catIds
per_class_ap50 = {}
for k, cid in enumerate(cat_ids):
    ap50_arr = ev.eval['precision'][0, :, k, 0, -1]
    ap50 = ap50_arr[ap50_arr > -1].mean() if (ap50_arr > -1).any() else float('nan')
    cname = CLASS_NAMES[cid]
    per_class_ap50[cname] = ap50
    print(f'  [{cid:2d}] {cname:22s}: AP50={ap50:.4f}')

assert mAP50 >= 0
assert per_class_ap50.get('person', 0) > 0 or per_class_ap50.get('car', 0) > 0, (
    'FAIL: AP50 is 0 for both person and car after 3 epochs -- category ids or bbox coords '
    'are almost certainly misaligned between predictions and ground truth.'
)
print('\nAlignment check PASSED: predictions and ground truth agree on ids/coordinates.')

In [ ]:
# ==============================================================================
# 6. FINAL SMOKE TEST SUMMARY
# ==============================================================================
print('='*70)
print('                     SMOKE TEST FINAL SUMMARY                          ')
print('='*70)
if train_run_info:
    print(f"Training throughput: {train_run_info['seconds_per_epoch']:.2f} s/epoch")
    print(f"Projected 100-epoch duration: {train_run_info['projected_100_epoch_hours']:.2f} hours")
    print(f"Training peak VRAM: {train_run_info['peak_vram_gb']:.2f} GB")
print(f'COCO predictions: {total_boxes} boxes exported')
print(f'mAP50: {mAP50:.4f} | mAP50-95: {mAP50_95:.4f}')
print(f"person AP50: {per_class_ap50.get('person', float('nan')):.4f}")
print(f"car AP50: {per_class_ap50.get('car', float('nan')):.4f}")
print('='*70)
print('>>> ALL SMOKE TEST CHECKS PASSED SUCCESSFULLY! <<<')
print('You are cleared to run full 100-epoch training (as 8 separate train+eval')
print('notebook pairs, same split pattern as this smoke test).')
print('='*70)